# 05 — Report Artifacts

Synthesize pipeline, data quality, and modeling outputs into report-ready tables and figures.

- Loads existing CSV/JSON artifacts only 
- Uses DuckDB where helpful; matplotlib for figures (no seaborn)
- Writes `reports/tables/`, `reports/figures/`, and `artifacts/manifests/run_manifest.json`

Run after notebooks `01`–`04` on the same `OPENF1_DATA_ROOT`.


## Colab setup (required every session)


In [1]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



Mounted at /content/drive
OPENF1_DATA_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
Cloning repository...
Working directory: /content/openf1-big-data-pipeline
PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts


## Load paths


In [2]:
from pathlib import Path

import pandas as pd

from openf1_pipeline.config import (
    get_artifacts_dir,
    get_data_quality_reports_dir,
    get_gold_dir,
    get_manifests_dir,
    get_model_results_dir,
    get_output_root,
    get_reports_dir,
)
from openf1_pipeline.reporting.report_tables import (
    build_bronze_endpoint_row_counts,
    build_confusion_matrix_table,
    build_data_volume_by_layer,
    build_error_analysis_summary_table,
    build_gold_feature_group_summary,
    build_gold_target_distribution_table,
    build_model_baseline_comparison_table,
    build_model_validation_test_metrics_table,
    build_reproducibility_artifacts_table,
    build_silver_cleaning_impact_table,
    build_silver_error_taxonomy_table,
    write_report_tables,
)
from openf1_pipeline.utils.cleanup import clean_report_artifacts

CLEAR_REPORT_ARTIFACTS = True

OUTPUT_ROOT = get_output_root()
DQ_DIR = get_data_quality_reports_dir()
MODEL_DIR = get_model_results_dir()
TABLES_DIR = get_reports_dir() / "tables"
FIGURES_DIR = get_reports_dir() / "figures"
MANIFESTS_DIR = get_manifests_dir()
ARTIFACTS_DIR = get_artifacts_dir()
GOLD_DIR = get_gold_dir()

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("CLEAR_REPORT_ARTIFACTS:", CLEAR_REPORT_ARTIFACTS)
print("TABLES_DIR:", TABLES_DIR)
print("FIGURES_DIR:", FIGURES_DIR)


OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
CLEAR_REPORT_ARTIFACTS: True
TABLES_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables
FIGURES_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports/figures


## Clean report artifacts


In [3]:
if CLEAR_REPORT_ARTIFACTS:
    print("Cleaning report tables and figures...")
    clean_report_artifacts(reports_dir=get_reports_dir())
else:
    print("Skipping report cleanup (CLEAR_REPORT_ARTIFACTS=False).")

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


INFO:openf1_pipeline.utils.io:Cleaned directory contents /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables (0 items removed)
INFO:openf1_pipeline.utils.io:Cleaned directory contents /content/drive/MyDrive/openf1_big_data_pipeline/reports/figures (0 items removed)
INFO:openf1_pipeline.utils.cleanup:Report artifacts cleanup complete: {'report_tables': 0, 'report_figures': 0}


Cleaning report tables and figures...


## Build report tables


In [4]:
gold_row_count = None
gold_mart = GOLD_DIR / "driver_race_feature_mart.parquet"
if gold_mart.exists():
    try:
        gold_row_count = len(pd.read_parquet(gold_mart, columns=["session_key"]))
    except Exception as exc:
        print("WARNING: could not read Gold mart row count:", exc)

tables = {
    "data_volume_by_layer": build_data_volume_by_layer(
        DQ_DIR / "bronze_row_counts.csv",
        DQ_DIR / "silver_table_inventory.csv",
        gold_row_count=gold_row_count,
    ),
    "bronze_endpoint_row_counts": build_bronze_endpoint_row_counts(DQ_DIR / "bronze_row_counts.csv"),
    "silver_cleaning_impact_table": build_silver_cleaning_impact_table(
        DQ_DIR / "silver_cleaning_impact_summary.csv"
    ),
    "silver_error_taxonomy_table": build_silver_error_taxonomy_table(
        DQ_DIR / "silver_duplicate_report.csv",
        DQ_DIR / "silver_outlier_report.csv",
        DQ_DIR / "silver_temporal_anomaly_report.csv",
        DQ_DIR / "silver_referential_integrity_report.csv",
    ),
    "gold_feature_group_summary": build_gold_feature_group_summary(
        ARTIFACTS_DIR / "feature_definitions" / "feature_dictionary.csv"
    ),
    "gold_target_distribution_table": build_gold_target_distribution_table(
        DQ_DIR / "gold_target_distribution.csv"
    ),
    "model_baseline_comparison_table": build_model_baseline_comparison_table(
        MODEL_DIR / "baseline_metrics.csv"
    ),
    "model_validation_test_metrics_table": build_model_validation_test_metrics_table(
        MODEL_DIR / "validation_metrics.csv",
        MODEL_DIR / "test_metrics.csv",
    ),
    "confusion_matrix_table": build_confusion_matrix_table(MODEL_DIR / "confusion_matrix.csv"),
    "error_analysis_summary_table": build_error_analysis_summary_table(
        MODEL_DIR / "error_analysis.csv"
    ),
    "reproducibility_artifacts_table": build_reproducibility_artifacts_table(
        MANIFESTS_DIR, ARTIFACTS_DIR, get_reports_dir()
    ),
}

table_paths = write_report_tables(tables, TABLES_DIR)
table_paths


INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/data_volume_by_layer.csv (21 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/bronze_endpoint_row_counts.csv (9 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/silver_cleaning_impact_table.csv (10 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/silver_error_taxonomy_table.csv (4 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/gold_feature_group_summary.csv (9 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/gold_target_distribution_table.csv (2 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/model_baseline_comparison_table.csv 

{'data_volume_by_layer': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/data_volume_by_layer.csv'),
 'bronze_endpoint_row_counts': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/bronze_endpoint_row_counts.csv'),
 'silver_cleaning_impact_table': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/silver_cleaning_impact_table.csv'),
 'silver_error_taxonomy_table': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/silver_error_taxonomy_table.csv'),
 'gold_feature_group_summary': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/gold_feature_group_summary.csv'),
 'gold_target_distribution_table': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/gold_target_distribution_table.csv'),
 'model_baseline_comparison_table': PosixPath('/content/drive/MyDrive/openf1_big_data_pipeline/reports/tables/model_baseline_comparison_table.csv'),
 'model_validat

## Architecture diagram placeholder


In [5]:
arch_md = FIGURES_DIR / "architecture_diagram_placeholder.md"
arch_md.write_text(
    '# Architecture diagram placeholder\n\n'
    'Insert Medallion pipeline diagram for the final report:\n\n'
    'OpenF1 API → Bronze JSONL → PySpark Bronze reports → DuckDB validation\n'
    '→ PySpark Silver → DuckDB → PySpark Gold → DuckDB → sklearn/LightGBM modeling\n',
    encoding="utf-8",
)
print("Wrote", arch_md)


Wrote /content/drive/MyDrive/openf1_big_data_pipeline/reports/figures/architecture_diagram_placeholder.md


## Figures (matplotlib only)


In [6]:
import matplotlib.pyplot as plt

# Target distribution
target_path = DQ_DIR / "gold_target_distribution.csv"
if target_path.is_file():
    td = pd.read_csv(target_path)
    if not td.empty and "points_finish" in td.columns:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar(td["points_finish"].astype(str), td["count"])
        ax.set_xlabel("points_finish")
        ax.set_ylabel("count")
        ax.set_title("Gold target distribution")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "target_distribution.png", dpi=120)
        plt.close(fig)
        print("Wrote target_distribution.png")
else:
    print("WARNING: gold_target_distribution.csv missing — skipped figure")

# Model metric comparison
metrics_path = MODEL_DIR / "validation_metrics.csv"
if metrics_path.is_file():
    metrics = pd.read_csv(metrics_path)
    if not metrics.empty and "f1" in metrics.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(metrics["model_name"], metrics["f1"])
        ax.set_ylabel("F1 (validation)")
        ax.set_title("Model metric comparison")
        plt.xticks(rotation=30, ha="right")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "model_metric_comparison.png", dpi=120)
        plt.close(fig)
        print("Wrote model_metric_comparison.png")
else:
    print("WARNING: validation_metrics.csv missing — skipped figure")

# Confusion matrix (first model in file)
cm_path = MODEL_DIR / "confusion_matrix.csv"
if cm_path.is_file():
    cm = pd.read_csv(cm_path)
    if not cm.empty:
        first_model = cm["model_name"].iloc[0]
        sub = cm[(cm["model_name"] == first_model) & (cm["split"] == cm["split"].iloc[0])]
        matrix = sub.pivot(index="actual", columns="predicted", values="count").fillna(0)
        fig, ax = plt.subplots(figsize=(5, 4))
        im = ax.imshow(matrix.values)
        ax.set_xticks(range(len(matrix.columns)))
        ax.set_yticks(range(len(matrix.index)))
        ax.set_xticklabels(matrix.columns)
        ax.set_yticklabels(matrix.index)
        ax.set_xlabel("predicted")
        ax.set_ylabel("actual")
        ax.set_title(f"Confusion matrix: {first_model}")
        fig.colorbar(im, ax=ax)
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=120)
        plt.close(fig)
        print("Wrote confusion_matrix.png")
else:
    print("WARNING: confusion_matrix.csv missing — skipped figure")

# Feature importance top 20
fi_path = MODEL_DIR / "feature_importance.csv"
if fi_path.is_file():
    fi = pd.read_csv(fi_path)
    if not fi.empty and "importance" in fi.columns:
        top = fi.sort_values("importance", ascending=False).head(20)
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.barh(top["feature_name"], top["importance"])
        ax.invert_yaxis()
        ax.set_title("Top 20 feature importances")
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "feature_importance_top20.png", dpi=120)
        plt.close(fig)
        print("Wrote feature_importance_top20.png")
else:
    print("WARNING: feature_importance.csv missing — skipped figure")

# Missingness before/after (top columns)
miss_before = DQ_DIR / "silver_missingness_before.csv"
miss_after = DQ_DIR / "silver_missingness_after.csv"
if miss_before.is_file() and miss_after.is_file():
    mb = pd.read_csv(miss_before)
    ma = pd.read_csv(miss_after)
    if not mb.empty and not ma.empty:
        top_cols = mb.sort_values("missing_pct", ascending=False).head(10)
        merged = top_cols.merge(
            ma[["table_name", "column_name", "missing_pct"]],
            on=["table_name", "column_name"],
            suffixes=("_before", "_after"),
        )
        labels = merged["table_name"] + "." + merged["column_name"]
        x = range(len(labels))
        width = 0.35
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar([i - width/2 for i in x], merged["missing_pct_before"], width, label="before")
        ax.bar([i + width/2 for i in x], merged["missing_pct_after"], width, label="after")
        ax.set_xticks(list(x))
        ax.set_xticklabels(labels, rotation=45, ha="right")
        ax.set_ylabel("missing_pct")
        ax.set_title("Silver missingness before vs after (top 10)")
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "missingness_before_after.png", dpi=120)
        plt.close(fig)
        print("Wrote missingness_before_after.png")
else:
    print("WARNING: silver missingness reports missing — skipped figure")


INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO:matplotlib.category:Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.


Wrote target_distribution.png
Wrote model_metric_comparison.png
Wrote confusion_matrix.png
Wrote feature_importance_top20.png
Wrote missingness_before_after.png


## Write run manifest


In [7]:
import json
from datetime import datetime, timezone

run_manifest = {
    "generated_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "output_root": str(OUTPUT_ROOT),
    "report_tables": {k: str(v) for k, v in table_paths.items()},
    "figures_dir": str(FIGURES_DIR),
    "note": "Paths only — populate row counts from Colab execution artifacts.",
}

model_manifest_path = MANIFESTS_DIR / "model_run_manifest.json"
if model_manifest_path.is_file():
    run_manifest["model_run_manifest"] = str(model_manifest_path)

manifest_out = MANIFESTS_DIR / "run_manifest.json"
manifest_out.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
print("Wrote", manifest_out)


Wrote /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/manifests/run_manifest.json
